In [8]:
from haystack import Pipeline
from haystack.components.generators.chat import OpenAIChatGenerator
from haystack.utils import Secret
from haystack.components.generators.utils import print_streaming_chunk
from chat2rag.config import CONFIG
from haystack_integrations.tools.mcp import MCPTool, SSEServerInfo
from haystack.components.tools import ToolInvoker
from haystack.dataclasses import ChatMessage

server_info = SSEServerInfo(url="http://localhost:8333/sse")
# server_info = SSEServerInfo(url="https://mcp.amap.com/sse?key=2502b472c2922df3c36c201bfc711018")
lead_way_tool = MCPTool(name="lead_way", server_info=server_info)

intention_llm = OpenAIChatGenerator(
    model="Qwen2.5-14B",
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    api_base_url=CONFIG.OPENAI_BASE_URL,
    streaming_callback=print_streaming_chunk,
    tools=[lead_way_tool],
)
generator_llm = OpenAIChatGenerator(
    model="Qwen2.5-32B",
    api_key=Secret.from_env_var("OPENAI_API_KEY"),
    api_base_url=CONFIG.OPENAI_BASE_URL,
    streaming_callback=print_streaming_chunk,
    tools=[lead_way_tool],
)
pipeline = Pipeline()
pipeline.add_component("intention_llm", intention_llm)
pipeline.add_component("tool_invoker", ToolInvoker(tools=[lead_way_tool]))
# pipeline.add_component("generator_llm", generator_llm)

pipeline.connect("intention_llm.replies", "tool_invoker.messages")
# pipeline.connect("tool_invoker.tool_messages", "generator_llm.messages")

🚅 Components
  - intention_llm: OpenAIChatGenerator
  - tool_invoker: ToolInvoker
🛤️ Connections
  - intention_llm.replies -> tool_invoker.messages (List[ChatMessage])

Error in sse_reader: peer closed connection without sending complete message body (incomplete chunked read)


In [10]:
import time
vin = "HTYW684948A352077"
current_time = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())
intent_prompt = f"""
你的设备码是{vin}
当前时间是{current_time}
你是工具意图识别助手。请分析用户输入并执行以下操作:

1. 如果识别到明确的工具/函数需求:
- 返回对应的工具及参数

2. 如果未识别到工具需求:
- 立即返回 None
- 不需要解释原因

注意:
- 只关注工具识别,忽略其他内容
- 快速决策,发现无匹配立即返回 None
- 不需要额外对话或解释
"""

history_messages = [
    ChatMessage.from_system(text=intent_prompt),
    ChatMessage.from_user(text="我想去洗手"),
]
pipeline.run({"intention_llm": {"messages": history_messages}})



[TOOL CALL]
Tool: lead_way 
Arguments:  {"vin": "HTYW684948A352077", "entity": "洗手间", "allow_nearest": false}



{'tool_invoker': {'tool_messages': [ChatMessage(_role=<ChatRole.TOOL: 'tool'>, _content=[ToolCallResult(result="meta=None content=[TextContent(type='text', text='任务执行成功，机器人回复：好的，我将带你去最近的卫生间', annotations=None)] isError=False", origin=ToolCall(tool_name='lead_way', arguments={'vin': 'HTYW684948A352077', 'entity': '洗手间', 'allow_nearest': False}, id='0197442c94843e9748c101593006955d'), error=False)], _name=None, _meta={})],
  'state': <haystack.components.agents.state.state.State at 0x1e144d20f10>}}